In [1]:
import pickle
from poke_env import RandomPlayer
from poke_env.data import GenData
from poke_env import AccountConfiguration
from poke_env.environment import SinglesEnv
from poke_env.environment.env import _EnvPlayer
from poke_env.battle import Battle
from poke_env.teambuilder import Teambuilder
from poke_env.player import Player
import numpy as np

In [2]:
current_game_save_path = "C:\Austin\Self_Projects\Pokemon_Sim\Reg_J_simulation_project\Singles_env_testing\env_stat_test_games\simu_game_test_2\Gen9OU-2025-11-13-smogonenvqcnpl-smogonenvmtqvz.pickle"
with open(current_game_save_path, 'rb') as handle:
    b = pickle.load(handle)

In [23]:
n=2

In [24]:
b["player1"][n]

In [25]:
n+=1
t0 = b["player1"][n]
t1 = b["player2"][n]

valid_action_mask = np.ones(10)

In [26]:
t0.current_observation # Usefull for game graph and masking unknown info

Observation(side_conditions={}, opponent_side_conditions={}, weather={<Weather.SUNNYDAY: 9>: 2}, fields={<Field.TRICK_ROOM: 11>: 1}, active_pokemon=ObservedPokemon(species='kingambit', level=100, name='Meruem', ability='supremeoverlord', boosts={'accuracy': 0, 'atk': 0, 'def': 0, 'evasion': 0, 'spa': 0, 'spd': 0, 'spe': 0}, current_hp_fraction=1.0, effects={}, is_dynamaxed=False, is_terastallized=False, item='leftovers', gender=<PokemonGender.FEMALE: 1>, moves=OrderedDict([('suckerpunch', suckerpunch (Move object)), ('ironhead', ironhead (Move object)), ('kowtowcleave', kowtowcleave (Move object)), ('swordsdance', swordsdance (Move object))]), tera_type=None, shiny=False, stats={'hp': 382, 'atk': 405, 'def': 276, 'spa': 140, 'spd': 206, 'spe': 159}, status=None), opponent_active_pokemon=ObservedPokemon(species='torkoal', level=100, name='Torkoal', ability=None, boosts={'accuracy': 0, 'atk': 0, 'def': 0, 'evasion': 0, 'spa': 0, 'spd': 0, 'spe': 0}, current_hp_fraction=1.0, effects={}, i

In [27]:
active_mon = t0.active_pokemon
full_team = [mon for mon in t0.team.values()]

In [8]:
valid_action_mask[6:] = np.array([(move in t0.available_moves) for move in active_mon.moves.values()],dtype=np.float64)
valid_action_mask
    

array([1., 1., 1., 1., 1., 1., 0., 1., 0., 0.])

In [28]:
t0.available_switches

[ironmoth (pokemon object) [Active: False, Status: None],
 deoxysspeed (pokemon object) [Active: False, Status: None],
 dragonite (pokemon object) [Active: False, Status: None],
 primarina (pokemon object) [Active: False, Status: None],
 greattusk (pokemon object) [Active: False, Status: None]]

In [29]:
full_team

[kingambit (pokemon object) [Active: True, Status: FNT],
 deoxysspeed (pokemon object) [Active: False, Status: None],
 greattusk (pokemon object) [Active: False, Status: None],
 dragonite (pokemon object) [Active: False, Status: None],
 ironmoth (pokemon object) [Active: False, Status: None],
 primarina (pokemon object) [Active: False, Status: None]]

In [30]:
valid_action_mask[:6] = np.array([(mon in t0.available_switches) for mon in full_team],dtype=np.float64)
valid_action_mask

array([0., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [31]:
np.random.choice(np.where(valid_action_mask==1.0)[0])

np.int64(2)

In [32]:
[mon for mon in t0.team.values()]

[kingambit (pokemon object) [Active: True, Status: FNT],
 deoxysspeed (pokemon object) [Active: False, Status: None],
 greattusk (pokemon object) [Active: False, Status: None],
 dragonite (pokemon object) [Active: False, Status: None],
 ironmoth (pokemon object) [Active: False, Status: None],
 primarina (pokemon object) [Active: False, Status: None]]

In [33]:
import random
from typing import List, Optional

from poke_env.battle.abstract_battle import AbstractBattle
from poke_env.battle.double_battle import DoubleBattle
from poke_env.battle.move_category import MoveCategory
from poke_env.battle.pokemon import Pokemon
from poke_env.battle.side_condition import SideCondition
from poke_env.battle.target import Target
from poke_env.player.battle_order import (
    BattleOrder,
    DefaultBattleOrder,
    DoubleBattleOrder,
    SingleBattleOrder,
)
from poke_env.player.player import Player

class Valid_Random_Player(Player):
    def choose_move(self, battle: AbstractBattle) -> BattleOrder:
        valid_action_mask = np.ones(10)
        action_list = [None, None, None, None, None, None, None, None, None, None]
        active_mon = battle.active_pokemon
        
        valid_action_mask[6:] = np.array([(move in battle.available_moves) for move in active_mon.moves.values()],dtype=np.float64)
        valid_action_mask[:6] = np.array([(mon in battle.available_switches) for mon in battle.team.values()],dtype=np.float64)

        action_list[:6] = [mon for mon in battle.team.values()]
        action_list[6:] = [move for move in active_mon.moves.values()]

        action = np.random.choice(np.where(valid_action_mask==1.0)[0])

        return self.create_order(action_list[action])

In [36]:
r = AccountConfiguration("acc2345","")
x = Valid_Random_Player(account_configuration=r)

In [42]:
print(valid_action_mask)
order = x.choose_move(t0)

[0. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [44]:
order.order.base_species

'dragonite'